In [ ]:
# Cài đặt thư viện TensorFlow Hub
!pip install -q tensorflow tensorflow-hub numpy

import tensorflow_hub as hub
import tensorflow as tf
import numpy as np

print("TensorFlow Version:", tf.__version__)
print("Môi trường đã sẵn sàng!")

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


TensorFlow Version: 2.21.0
Môi trường đã sẵn sàng!


In [ ]:
# Tải mô hình ELMo v3 từ TF-Hub
print("Đang tải mô hình ELMo v3...")
elmo = hub.load("https://tfhub.dev/google/elmo/3")
print("-> Đã tải xong mô hình ELMo!")

Đang tải mô hình ELMo v3...
-> Đã tải xong mô hình ELMo!


In [ ]:
# thực nghiệm đo độ tương đồng theo ngữ cảnh
# 1. Định nghĩa 2 câu có từ "bank" mang nghĩa khác nhau
sentences = [
    "He went to the bank to withdraw money.",
    "He sat on the bank of the river."
]
#"Anh ấy đã đến ngân hàng để rút tiền.",
# "Anh ấy ngồi bên bờ sông."

# 2. Hàm trích xuất embedding từ ELMo
def elmo_embed(sentence):
    # elmo.signatures["default"] trả về các biểu diễn của ELMo
    embeddings = elmo.signatures["default"](tf.constant([sentence]))["elmo"]
    # Tính vector trung bình cho toàn bộ câu (sentence-level representation)
    return tf.reduce_mean(embeddings, axis=1).numpy()

# 3. Lấy vector của từng câu
vec1 = elmo_embed(sentences[0])
vec2 = elmo_embed(sentences[1])

# 4. Tính Cosine Similarity giữa 2 vector
cos_sim = np.dot(vec1, vec2.T) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print(f"Câu 1: '{sentences[0]}'")
print(f"Câu 2: '{sentences[1]}'")
print(f"\n Độ tương đồng Cosine giữa hai câu: {cos_sim[0][0]:.4f}")

Câu 1: 'He went to the bank to withdraw money.'
Câu 2: 'He sat on the bank of the river.'

 Độ tương đồng Cosine giữa hai câu: 0.5557


In [ ]:
# thực nghiệm đo độ tương đồng theo ngữ cảnh
# 1. Định nghĩa 2 câu có từ "bank" mang nghĩa khác nhau
sentences = [
    "Apple released a new smartphone yesterday.",
    "I ate a sweet red apple for breakfast."
]
#"Apple đã ra mắt một chiếc điện thoại thông minh mới vào hôm qua.",
# "Tôi đã ăn một quả táo đỏ ngọt vào bữa sáng."

# 2. Hàm trích xuất embedding từ ELMo
def elmo_embed(sentence):
    # elmo.signatures["default"] trả về các biểu diễn của ELMo
    embeddings = elmo.signatures["default"](tf.constant([sentence]))["elmo"]
    # Tính vector trung bình cho toàn bộ câu (sentence-level representation)
    return tf.reduce_mean(embeddings, axis=1).numpy()

# 3. Lấy vector của từng câu
vec1 = elmo_embed(sentences[0])
vec2 = elmo_embed(sentences[1])

# 4. Tính Cosine Similarity giữa 2 vector
cos_sim = np.dot(vec1, vec2.T) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print(f"Câu 1: '{sentences[0]}'")
print(f"Câu 2: '{sentences[1]}'")
print(f"\n Độ tương đồng Cosine giữa hai câu: {cos_sim[0][0]:.4f}")

Câu 1: 'Apple released a new smartphone yesterday.'
Câu 2: 'I ate a sweet red apple for breakfast.'

 Độ tương đồng Cosine giữa hai câu: 0.3321


In [ ]:
# Cài đặt Hugging Face Transformers và PyTorch
!pip install -q transformers torch

import torch
from transformers import BertTokenizer, BertForMaskedLM, BertModel
import numpy as np

print("PyTorch Version:", torch.__version__)
print("-> Sẵn sàng!")

PyTorch Version: 2.11.0+cpu
-> Sẵn sàng!


In [ ]:
# THỰC NGHIỆM MASKED LANGUAGE MODEL (MLM)
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForMaskedLM

# 1. Tải mô hình và bộ tách từ (Tokenizer) của BERT
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)
model.eval()

sentence = "Paris is the [MASK] of France."
#Paris là [MASK] của Pháp.

# Kiểm tra xem câu có token [MASK] hay không
if "[MASK]" not in sentence:
    raise ValueError("Lỗi: Câu đầu vào bắt buộc phải chứa token '[MASK]' để BERT đoán!")

# 3. Mã hóa câu văn bản thành Tensor đầu vào
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # Kích thước: [batch_size, seq_len, vocab_size]

# 4. Tự động xác định vị trí của [MASK] trong chuỗi token
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1][0]
mask_logits = logits[0, mask_token_index]

# 5. Dùng Softmax để chuyển đổi điểm số (logits) thành phần trăm xác suất (%)
probabilities = F.softmax(mask_logits, dim=-1)

# Lấy Top 5 từ có xác suất cao nhất
top_k = 5
top_probs, top_indices = torch.topk(probabilities, top_k)

print(f"Câu gốc đầu vào: '{sentence}'\n")
print(f"{'Hạng':<6} | {'Từ dự đoán':<15} | {'Xác suất (%)':<15} | {'Câu hoàn chỉnh'}")
print("-" * 75)

for i in range(top_k):
    token_id = top_indices[i].item()
    prob = top_probs[i].item() * 100
    predicted_word = tokenizer.decode([token_id]).strip()

    # Thay thế [MASK] bằng từ vừa đoán được để tạo câu hoàn chỉnh
    completed_sentence = sentence.replace("[MASK]", f"[{predicted_word}]")
    print(f"Top {i+1:<2} | {predicted_word:<15} | {prob:>6.2f}%         | {completed_sentence}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Câu gốc đầu vào: 'Paris is the [MASK] of France.'

Hạng   | Từ dự đoán      | Xác suất (%)    | Câu hoàn chỉnh
---------------------------------------------------------------------------
Top 1  | capital         |  99.69%         | Paris is the [capital] of France.
Top 2  | heart           |   0.06%         | Paris is the [heart] of France.
Top 3  | center          |   0.04%         | Paris is the [center] of France.
Top 4  | centre          |   0.03%         | Paris is the [centre] of France.
Top 5  | city            |   0.03%         | Paris is the [city] of France.


In [ ]:
# THỰC NGHIỆM MASKED LANGUAGE MODEL (MLM)
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForMaskedLM

# 1. Tải mô hình và bộ tách từ (Tokenizer) của BERT
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)
model.eval()

sentence = "I enjoy drinking a hot cup of [MASK] in the morning."
#Tôi thích uống một tách [MASK] nóng vào buổi sáng.

# Kiểm tra xem câu có token [MASK] hay không
if "[MASK]" not in sentence:
    raise ValueError("Lỗi: Câu đầu vào bắt buộc phải chứa token '[MASK]' để BERT đoán!")

# 3. Mã hóa câu văn bản thành Tensor đầu vào
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # Kích thước: [batch_size, seq_len, vocab_size]

# 4. Tự động xác định vị trí của [MASK] trong chuỗi token
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1][0]
mask_logits = logits[0, mask_token_index]

# 5. Dùng Softmax để chuyển đổi điểm số (logits) thành phần trăm xác suất (%)
probabilities = F.softmax(mask_logits, dim=-1)

# Lấy Top 5 từ có xác suất cao nhất
top_k = 5
top_probs, top_indices = torch.topk(probabilities, top_k)

print(f"Câu gốc đầu vào: '{sentence}'\n")
print(f"{'Hạng':<6} | {'Từ dự đoán':<15} | {'Xác suất (%)':<15} | {'Câu hoàn chỉnh'}")
print("-" * 75)

for i in range(top_k):
    token_id = top_indices[i].item()
    prob = top_probs[i].item() * 100
    predicted_word = tokenizer.decode([token_id]).strip()

    # Thay thế [MASK] bằng từ vừa đoán được để tạo câu hoàn chỉnh
    completed_sentence = sentence.replace("[MASK]", f"[{predicted_word}]")
    print(f"Top {i+1:<2} | {predicted_word:<15} | {prob:>6.2f}%         | {completed_sentence}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Câu gốc đầu vào: 'I enjoy drinking a hot cup of [MASK] in the morning.'

Hạng   | Từ dự đoán      | Xác suất (%)    | Câu hoàn chỉnh
---------------------------------------------------------------------------
Top 1  | coffee          |  86.32%         | I enjoy drinking a hot cup of [coffee] in the morning.
Top 2  | tea             |   7.41%         | I enjoy drinking a hot cup of [tea] in the morning.
Top 3  | water           |   2.33%         | I enjoy drinking a hot cup of [water] in the morning.
Top 4  | beer            |   0.51%         | I enjoy drinking a hot cup of [beer] in the morning.
Top 5  | wine            |   0.50%         | I enjoy drinking a hot cup of [wine] in the morning.


In [ ]:
# Nạp thư viện và mô hình GPT-2 của OpenAI
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
print("-> Đã nạp xong GPT-2!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

-> Đã nạp xong GPT-2!


In [ ]:
# Đưa mồi câu cho GPT tự viết tiếp
prompt = "Artificial Intelligence will change"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# GPT tự động sinh tiếp 15 từ tiếp theo
#PT viết tiếp thành: Trí tuệ nhân tạo sẽ thay đổi cách chúng ta suy nghĩ về thế giới.

# Tương lai của AI là
output = model.generate(input_ids, max_length=20, pad_token_id=tokenizer.eos_token_id)
result = tokenizer.decode(output[0])

print("Câu mồi:", prompt)
print("GPT viết tiếp thành:", result)

Câu mồi: Artificial Intelligence will change
GPT viết tiếp thành: Artificial Intelligence will change the way we think about the world.

The future of AI is


In [ ]:
# Cài đặt thư viện Scikit-learn, Gensim và khởi tạo dữ liệu
!pip install -q scikit-learn gensim pandas numpy

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec, FastText
import gensim.downloader as api

# BẠN CÓ THỂ ĐỔI BẤT KỲ CÂU GÌ VÀO ĐÂY (CẢ TIẾNG VIỆT LẪN TIẾNG ANH)
sample_docs = [
    "tôi thích học xử lý ngôn ngữ tự nhiên",
    "xử lý ngôn ngữ tự nhiên rất thú vị",
    "tôi thích nghiên cứu trí tuệ nhân tạo"
]
print("-> Đã sẵn sàng dữ liệu thử nghiệm!")

-> Đã sẵn sàng dữ liệu thử nghiệm!


In [ ]:
#  Demo One-Hot Encoding
vocab = sorted(list(set(" ".join(sample_docs).split())))
vocab_size = len(vocab)
word_to_ohe = {w: np.eye(vocab_size)[i] for i, w in enumerate(vocab)}

# Thử lấy vector One-Hot của 1 từ
tu_thu = "học"
print(f"1. Kích thước vector One-Hot: {vocab_size} chiều (|V|)")
print(f"2. Vector của từ '{tu_thu}':\n", word_to_ohe[tu_thu].astype(int))

# Chứng minh tính trực giao (tích vô hướng = 0, không đo được ngữ nghĩa)
dot_sim = np.dot(word_to_ohe["học"], word_to_ohe["thích"])
print(f"\n-> Tích vô hướng giữa 'học' và 'thích': {dot_sim} (Hoàn toàn trực giao, triệt tiêu ngữ nghĩa!)")

1. Kích thước vector One-Hot: 18 chiều (|V|)
2. Vector của từ 'học':
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

-> Tích vô hướng giữa 'học' và 'thích': 0.0 (Hoàn toàn trực giao, triệt tiêu ngữ nghĩa!)


In [ ]:
#  Demo Bag-of-Words (CountVectorizer)
bow_vec = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
bow_matrix = bow_vec.fit_transform(sample_docs)

# Hiển thị ma trận đếm dạng bảng
df_bow = pd.DataFrame(bow_matrix.toarray(), columns=bow_vec.get_feature_names_out(), index=["Câu 1", "Câu 2", "Câu 3"])
print("Ma trận Bag-of-Words (Đếm số lần xuất hiện):")
display(df_bow)

Ma trận Bag-of-Words (Đếm số lần xuất hiện):


,cứu,học,lý,nghiên,ngôn,ngữ,nhiên,nhân,rất,thích,thú,trí,tuệ,tôi,tạo,tự,vị,xử
Câu 1,0,1,1,0,1,1,1,0,0,1,0,0,0,1,0,1,0,1
Câu 2,0,0,1,0,1,1,1,0,1,0,1,0,0,0,0,1,1,1
Câu 3,1,0,0,1,0,0,0,1,0,1,0,1,1,1,1,0,0,0


In [ ]:
# Demo TF-IDF
tfidf_vec = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
tfidf_matrix = tfidf_vec.fit_transform(sample_docs)

df_tfidf = pd.DataFrame(np.round(tfidf_matrix.toarray(), 3), columns=tfidf_vec.get_feature_names_out(), index=["Câu 1", "Câu 2", "Câu 3"])
print("Ma trận trọng số TF-IDF:")
display(df_tfidf)
print("\n-> Nhận xét: Từ 'xử', 'lý', 'ngôn', 'ngữ' xuất hiện ở cả Câu 1 & 2 nên điểm TF-IDF bị giảm xuống.")

Ma trận trọng số TF-IDF:


,cứu,học,lý,nghiên,ngôn,ngữ,nhiên,nhân,rất,thích,thú,trí,tuệ,tôi,tạo,tự,vị,xử
Câu 1,0.000,0.422,0.321,0.000,0.321,0.321,0.321,0.000,0.000,0.321,0.000,0.000,0.000,0.321,0.000,0.321,0.000,0.321
Câu 2,0.000,0.000,0.299,0.000,0.299,0.299,0.299,0.000,0.393,0.000,0.393,0.000,0.000,0.000,0.000,0.299,0.393,0.299
Câu 3,0.374,0.000,0.000,0.374,0.000,0.000,0.000,0.374,0.000,0.284,0.000,0.374,0.374,0.284,0.374,0.000,0.000,0.000



-> Nhận xét: Từ 'xử', 'lý', 'ngôn', 'ngữ' xuất hiện ở cả Câu 1 & 2 nên điểm TF-IDF bị giảm xuống.


In [ ]:
# Demo Word2Vec (Skip-gram, sg=1)
corpus_tokens = [s.split() for s in sample_docs]
w2v_model = Word2Vec(corpus_tokens, vector_size=50, window=3, min_count=1, sg=1, epochs=150, seed=42)

target_w = "ngôn"
print(f"1. Kích thước vector của từ '{target_w}': {w2v_model.wv[target_w].shape} (Vector thực dày đặc)")
print("2. 5 con số đầu tiên trong vector:", w2v_model.wv[target_w][:5])

print(f"\n3. Các từ có ngữ cảnh xuất hiện gần nhất với '{target_w}':")
for w, score in w2v_model.wv.most_similar(target_w, topn=3):
    print(f"  - {w}: {score:.4f}")

1. Kích thước vector của từ 'ngôn': (50,) (Vector thực dày đặc)
2. 5 con số đầu tiên trong vector: [ 0.01451195 -0.00819994  0.01652356 -0.00807085 -0.00179007]

3. Các từ có ngữ cảnh xuất hiện gần nhất với 'ngôn':
  - cứu: 0.4102
  - tuệ: 0.2982
  - học: 0.2811


In [ ]:
# Demo GloVe (Tải vector tiền huấn luyện glove-twitter-25)
print("Đang tải vector GloVe nhẹ (25 chiều)...")
glove = api.load("glove-twitter-25")

# Kiểm chứng phép toán đại số vector: king - man + woman = ?
result = glove.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
print("\nMinh họa đại số vector GloVe (king - man + woman):")
print(f"-> Kết quả dự đoán hàng đầu: '{result[0][0]}' với Cosine similarity = {result[0][1]:.4f}")

Đang tải vector GloVe nhẹ (25 chiều)...
[==================================================] 100.0% 104.8/104.8MB downloaded

Minh họa đại số vector GloVe (king - man + woman):
-> Kết quả dự đoán hàng đầu: 'meets' với Cosine similarity = 0.8842


In [ ]:
#  Demo FastText & Thử nghiệm từ OOV
ft_model = FastText(corpus_tokens, vector_size=50, window=3, min_count=1, min_n=2, max_n=4, epochs=150, seed=42)

# Đưa vào một từ ghép mới lạ hoàn toàn chưa có trong dữ liệu huấn luyện
oov_word = "ngôn_từ"

print(f"Kiểm tra từ OOV: '{oov_word}'")
print(f"- Có trong từ điển Word2Vec không? -> {oov_word in w2v_model.wv.key_to_index}")
print(f"- Có trong từ điển FastText không? -> {oov_word in ft_model.wv.key_to_index}")

# Thử lấy vector bằng Word2Vec -> Văng lỗi KeyError
try:
    _ = w2v_model.wv[oov_word]
except KeyError:
    print("\n[Word2Vec]: Báo lỗi KeyError! (Bất lực vì từ chưa từng học)")

# Thử lấy vector bằng FastText -> Thành công nhờ các subwords
oov_vec = ft_model.wv[oov_word]
print(f"\n[FastText]: Tạo vector thành công nhờ Subwords! (Kích thước: {oov_vec.shape})")
print(f"Từ gần nhất trong ngữ liệu với '{oov_word}': {ft_model.wv.most_similar(oov_word, topn=1)[0]}")

In [1]:
# [CELL 1]: Cài đặt Hugging Face Transformers, PyTorch và thư viện tách từ tiếng Việt
!pip install -q transformers torch pyvi

import torch
from transformers import AutoTokenizer, AutoModel
from pyvi import ViTokenizer

# Kiểm tra xem GPU đã sẵn sàng chưa
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-> Môi trường tính toán đang sử dụng: {device}")

-> Môi trường tính toán đang sử dụng: cuda


In [2]:
# [CELL 2]: Tải PhoBERT-base-v2 chính thức của VinAI Research
MODEL_NAME = "vinai/phobert-base-v2"

print(f"Đang tải mô hình {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()  # Chuyển sang chế độ trích xuất đặc trưng (evaluation)

print("-> [XONG Ô 2]: Đã tải và nạp xong PhoBERT vào GPU!")

Đang tải mô hình vinai/phobert-base-v2...


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


-> [XONG Ô 2]: Đã tải và nạp xong PhoBERT vào GPU!


In [3]:
# [CELL 3]: Chạy kiểm thử mã hóa câu tiếng Việt với PhoBERT
raw_sentence = "Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên."

print(f"1. Câu văn gốc ban đầu:\n   '{raw_sentence}'\n")

# BƯỚC 1: Tiền xử lý tách từ ghép tiếng Việt (Bắt buộc với PhoBERT)
segmented_sentence = ViTokenizer.tokenize(raw_sentence)
print(f"2. Sau khi chạy qua bộ tách từ ghép (ViTokenizer):\n   '{segmented_sentence}'\n")

# BƯỚC 2: Tokenize thành các Subwords theo từ điển 64.000 từ của PhoBERT
tokens = tokenizer.tokenize(segmented_sentence)
print(f"3. Danh sách các token/subword PhoBERT nhận diện:")
print(f"   {tokens}\n")

# BƯỚC 3: Chuyển thành Tensor IDs và đưa qua mạng Transformer Encoder
inputs = tokenizer(segmented_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    # Lấy ma trận vector của tất cả các từ trong câu (Hidden States)
    last_hidden_state = outputs.last_hidden_state
    # Lấy vector đặc trưng ngữ cảnh đại diện cho toàn bộ câu (tại vị trí token [CLS] đầu tiên)
    cls_vector = last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(f"4. Kích thước ma trận ngữ cảnh của cả câu (Batch, Số token, Số chiều):")
print(f"   {last_hidden_state.shape}")
print(f"5. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):")
print(f"   {cls_vector.shape} chiều (Chuẩn RoBERTa-base: 768 chiều)")
print(f"6. 5 giá trị số thực đầu tiên của vector [CLS]:")
print(f"   {cls_vector[:5]}")

1. Câu văn gốc ban đầu:
   'Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên.'

2. Sau khi chạy qua bộ tách từ ghép (ViTokenizer):
   'Học_sinh sinh_viên nghiên_cứu xử_lý ngôn_ngữ tự_nhiên .'

3. Danh sách các token/subword PhoBERT nhận diện:
   ['Học_sinh', 'sinh_viên', 'nghiên_cứu', 'xử_lý', 'ngôn_ngữ', 'tự_nhiên', '.']

4. Kích thước ma trận ngữ cảnh của cả câu (Batch, Số token, Số chiều):
   torch.Size([1, 9, 768])
5. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):
   (768,) chiều (Chuẩn RoBERTa-base: 768 chiều)
6. 5 giá trị số thực đầu tiên của vector [CLS]:
   [ 0.04448306  0.32333156 -0.295708   -0.6966495  -0.2551121 ]


In [4]:
# [CELL 1 - viBERT]: Nạp mô hình viBERT-base-cased chính thức từ FPT AI
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đường dẫn mô hình viBERT chuẩn trên Hugging Face
VIBERT_MODEL_NAME = "FPTAI/vibert-base-cased"

print(f"Đang tải mô hình {VIBERT_MODEL_NAME}...")
vibert_tokenizer = AutoTokenizer.from_pretrained(VIBERT_MODEL_NAME)
vibert_model = AutoModel.from_pretrained(VIBERT_MODEL_NAME).to(device)
vibert_model.eval()

print("-> [XONG BƯỚC 1]: Đã tải thành công viBERT vào bộ nhớ GPU!")

Đang tải mô hình FPTAI/vibert-base-cased...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  581MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  581MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: FPTAI/vibert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> [XONG BƯỚC 1]: Đã tải thành công viBERT vào bộ nhớ GPU!


In [5]:
# [CELL 2 - viBERT]: Kiểm nghiệm băm Subword và trích xuất đặc trưng
test_sentence = "Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên."

print(f"1. Câu văn gốc đưa vào (Không cần tách từ trước):\n   '{test_sentence}'\n")

# BƯỚC 1: WordPiece tự động băm nhỏ câu thành các tokens / subwords
tokens = vibert_tokenizer.tokenize(test_sentence)
print(f"2. Danh sách tokens do WordPiece của viBERT phân tách:")
print(f"   {tokens}\n")

# BƯỚC 2: Chuyển tokens thành Tensor IDs đưa vào mạng Transformer
inputs = vibert_tokenizer(test_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = vibert_model(**inputs)
    # Ma trận biểu diễn toàn bộ các token trong câu
    last_hidden_state = outputs.last_hidden_state
    # Vector đặc trưng ngữ cảnh đại diện toàn câu (tại token [CLS] đầu tiên)
    cls_vector = last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(f"3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):")
print(f"   {last_hidden_state.shape}")
print(f"4. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):")
print(f"   {cls_vector.shape} chiều (Chuẩn BERT-base: 768 chiều)")
print(f"5. 5 phần tử số thực đầu tiên của vector [CLS]:")
print(f"   {cls_vector[:5]}")

1. Câu văn gốc đưa vào (Không cần tách từ trước):
   'Học sinh sinh viên nghiên cứu xử lý ngôn ngữ tự nhiên.'

2. Danh sách tokens do WordPiece của viBERT phân tách:
   ['hoc', 'sinh', 'sinh', 'vien', 'nghi', '##en', 'cu', '##u', 'x', '##u', 'ly', 'ng', '##on', 'ng', '##u', 'tu', 'n', '##hien', '.']

3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):
   torch.Size([1, 21, 768])
4. Kích thước vector đại diện ngữ cảnh toàn câu ([CLS]):
   (768,) chiều (Chuẩn BERT-base: 768 chiều)
5. 5 phần tử số thực đầu tiên của vector [CLS]:
   [ 0.02030429  0.23482998  0.2288707  -0.7599542   0.25660804]


In [6]:
# [CELL 1 - XLM-RoBERTa]: Tải mô hình xlm-roberta-base chính thức
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tên mô hình chính thức trên Hugging Face
XLM_MODEL_NAME = "xlm-roberta-base"

print(f"Đang tải {XLM_MODEL_NAME} (khoảng 1.1GB, mất tầm 30-45 giây)...")
xlm_tokenizer = AutoTokenizer.from_pretrained(XLM_MODEL_NAME)
xlm_model = AutoModel.from_pretrained(XLM_MODEL_NAME).to(device)
xlm_model.eval()

print("-> [XONG BƯỚC 1]: Đã tải thành công XLM-RoBERTa vào GPU!")

Đang tải xlm-roberta-base (khoảng 1.1GB, mất tầm 30-45 giây)...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> [XONG BƯỚC 1]: Đã tải thành công XLM-RoBERTa vào GPU!


In [7]:
# [CELL 2 - XLM-RoBERTa]: Kiểm nghiệm SentencePiece trên câu pha trộn Anh - Việt (Code-mixing)
test_sentence = "Áo này order về mặc bao cool ngầu, feedback 5 sao nha shop!"

print(f"1. Câu kiểm thử (Code-mixing thực tế trên E-commerce):\n   '{test_sentence}'\n")

# BƯỚC 1: Phân tách Subwords bằng SentencePiece (Từ điển 250k)
tokens = xlm_tokenizer.tokenize(test_sentence)
print(f"2. Danh sách Subwords do SentencePiece băm ra:")
print(f"   {tokens}\n")

# BƯỚC 2: Chuyển chuỗi thành Tensor đưa vào mạng Transformer
inputs = xlm_tokenizer(test_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = xlm_model(**inputs)
    last_hidden_state = outputs.last_hidden_state
    # Trích xuất vector biểu diễn ngữ cảnh toàn câu tại token đầu tiên [CLS]
    cls_vector = last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(f"3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):")
print(f"   {last_hidden_state.shape}")
print(f"4. Kích thước vector đặc trưng toàn câu ([CLS]):")
print(f"   {cls_vector.shape} chiều (Chuẩn RoBERTa: 768 chiều)")
print(f"5. 5 giá trị số thực đầu tiên của vector [CLS]:")
print(f"   {cls_vector[:5]}")

1. Câu kiểm thử (Code-mixing thực tế trên E-commerce):
   'Áo này order về mặc bao cool ngầu, feedback 5 sao nha shop!'

2. Danh sách Subwords do SentencePiece băm ra:
   ['▁Á', 'o', '▁này', '▁order', '▁về', '▁mặc', '▁bao', '▁cool', '▁ng', 'ầ', 'u', ',', '▁feedback', '▁5', '▁sao', '▁nha', '▁shop', '!']

3. Kích thước ma trận ngữ cảnh đầu ra (Batch, Tokens, Dimensions):
   torch.Size([1, 20, 768])
4. Kích thước vector đặc trưng toàn câu ([CLS]):
   (768,) chiều (Chuẩn RoBERTa: 768 chiều)
5. 5 giá trị số thực đầu tiên của vector [CLS]:
   [ 0.07837722  0.05659284  0.09238684 -0.00107576 -0.02136643]
